In [1]:
import pandas as pd
import re
from collections import defaultdict

# Load the meta cluster coverage dataframe
coverage_df = pd.read_csv('artifacts/pbr/meta_cluster_coverage.csv', index_col=0)

# Display the head
print("Shape:", coverage_df.shape)
print("\nHead:")
coverage_df.head()

Shape: (85, 6)

Head:


,crawler_log_20251127_083333_DeepSeek-R1-Distill-Llama-8B_1samples_100crawls_Truefilter_thought_prefill_with_seedprompt_vllm.json,crawler_log_20251127_113804_DeepSeek-R1-Distill-Llama-8B_1samples_100crawls_Truefilter_user_prefill_with_seedprompt_vllm.json,crawler_log_20251127_200627_Llama-3.1-Tulu-3-8B-SFT_1samples_100crawls_Truefilter_assistant_prefill_with_seedprompt_vllm.json,crawler_log_20251128_103118_Llama-3.1-8B-Instruct_1samples_100crawls_Truefilter_assistant_prefill_with_seedprompt_vllm.json,crawler_log_20251128_141522_Llama-3.1-Tulu-3-8B-SFT_1samples_100crawls_Truefilter_user_prefill_with_seedprompt_vllm.json,crawler_log_20251130_122103_Llama-3.1-8B-Instruct_1samples_100crawls_Truefilter_user_prefill_with_seedprompt_vllm.json
2021 US capitol insurrection,False,False,False,False,True,False
AI assistant does not discuss the future,True,True,True,False,True,False
AI assistant does not have access to location,False,False,True,False,False,False
AI assistant does not have opinions or experiences,False,True,True,False,True,False
AI assistant has knowledge cutoff and no access to real time information,False,False,True,False,True,False


In [2]:
def parse_crawler_log_filename(filename: str):
    """Parse crawler log filename to extract model name."""
    basename = filename.replace(".json", "")
    
    # Try pattern with "prefill" separator
    pattern = r"crawler_log_\d{8}_\d{6}_(.+?)_\d+samples_\d+crawls_.+?filter_(.+?)_prefill_.+"
    match = re.match(pattern, basename)
    
    if match:
        model = match.group(1)
        prefill_mode = match.group(2)
        return {"model": model, "prefill_mode": prefill_mode}
    return None

# Group columns by model
model_columns = defaultdict(list)

for col in coverage_df.columns:
    parsed = parse_crawler_log_filename(col)
    if parsed:
        model_columns[parsed["model"]].append(col)
    else:
        print(f"Warning: Could not parse column: {col}")

# Display column names per model
print("Column names per model:\n")
for model, cols in sorted(model_columns.items()):
    print(f"{model}:")
    for col in cols:
        parsed = parse_crawler_log_filename(col)
        print(f"  - {col} (prefill: {parsed['prefill_mode'] if parsed else 'unknown'})")
    print()

Column names per model:

DeepSeek-R1-Distill-Llama-8B:
  - crawler_log_20251127_083333_DeepSeek-R1-Distill-Llama-8B_1samples_100crawls_Truefilter_thought_prefill_with_seedprompt_vllm.json (prefill: thought)
  - crawler_log_20251127_113804_DeepSeek-R1-Distill-Llama-8B_1samples_100crawls_Truefilter_user_prefill_with_seedprompt_vllm.json (prefill: user)

Llama-3.1-8B-Instruct:
  - crawler_log_20251128_103118_Llama-3.1-8B-Instruct_1samples_100crawls_Truefilter_assistant_prefill_with_seedprompt_vllm.json (prefill: assistant)
  - crawler_log_20251130_122103_Llama-3.1-8B-Instruct_1samples_100crawls_Truefilter_user_prefill_with_seedprompt_vllm.json (prefill: user)

Llama-3.1-Tulu-3-8B-SFT:
  - crawler_log_20251127_200627_Llama-3.1-Tulu-3-8B-SFT_1samples_100crawls_Truefilter_assistant_prefill_with_seedprompt_vllm.json (prefill: assistant)
  - crawler_log_20251128_141522_Llama-3.1-Tulu-3-8B-SFT_1samples_100crawls_Truefilter_user_prefill_with_seedprompt_vllm.json (prefill: user)



In [3]:
# Store model columns dictionary for easy access
model_columns

defaultdict(list,
            {'DeepSeek-R1-Distill-Llama-8B': ['crawler_log_20251127_083333_DeepSeek-R1-Distill-Llama-8B_1samples_100crawls_Truefilter_thought_prefill_with_seedprompt_vllm.json',
              'crawler_log_20251127_113804_DeepSeek-R1-Distill-Llama-8B_1samples_100crawls_Truefilter_user_prefill_with_seedprompt_vllm.json'],
             'Llama-3.1-Tulu-3-8B-SFT': ['crawler_log_20251127_200627_Llama-3.1-Tulu-3-8B-SFT_1samples_100crawls_Truefilter_assistant_prefill_with_seedprompt_vllm.json',
              'crawler_log_20251128_141522_Llama-3.1-Tulu-3-8B-SFT_1samples_100crawls_Truefilter_user_prefill_with_seedprompt_vllm.json'],
             'Llama-3.1-8B-Instruct': ['crawler_log_20251128_103118_Llama-3.1-8B-Instruct_1samples_100crawls_Truefilter_assistant_prefill_with_seedprompt_vllm.json',
              'crawler_log_20251130_122103_Llama-3.1-8B-Instruct_1samples_100crawls_Truefilter_user_prefill_with_seedprompt_vllm.json']})

In [4]:
# Create a Series per model: True if ANY of the model's columns has True for that meta cluster
model_series = {
    m: coverage_df[model_columns[m]].any(axis=1) for m in model_columns
}

# Print each model's series
for model, series in sorted(model_series.items()):
    print(f"\n{'='*80}")
    print(f"Model: {model}")
    print(f"{'='*80}")
    print(series.head(10))
    print(f"\nTotal True values: {series.sum()} out of {len(series)}")


Model: DeepSeek-R1-Distill-Llama-8B
2021 US capitol insurrection                                                False
AI assistant does not discuss the future                                     True
AI assistant does not have access to location                               False
AI assistant does not have opinions or experiences                           True
AI assistant has knowledge cutoff and no access to real time information    False
AI assistant is inable to provide professional advice                       False
AI assistant is not embodied in the real world                              False
CSAM                                                                        False
China-India border conflict                                                  True
China-Japan relations                                                        True
dtype: bool

Total True values: 51 out of 85

Model: Llama-3.1-8B-Instruct
2021 US capitol insurrection                                        

In [6]:
# Compute recall for each column against its model's union series
recall_results = []

for col in coverage_df.columns:
    # Get the column's coverage
    col_series = coverage_df[col]
    
    # Find which model this column belongs to
    parsed = parse_crawler_log_filename(col)
    if not parsed:
        continue
    
    model = parsed["model"]
    prefill_mode = parsed["prefill_mode"]
    
    # Get the model's union series (ground truth)
    model_ground_truth = model_series[model]
    
    # Compute TP (True Positives): covered by both column and model union
    tp = ((col_series == True) & (model_ground_truth == True)).sum()
    
    # Total positives in ground truth (TP + FN)
    total_positives = model_ground_truth.sum()
    
    # Compute recall
    if total_positives > 0:
        recall = tp / total_positives
    else:
        recall = 0.0
    
    recall_results.append({
        # 'column': col,
        'model': model,
        'prefill_mode': prefill_mode,
        'tp': tp,
        'total_positives': total_positives,
        'recall': recall
    })

# Create DataFrame with results
recall_df = pd.DataFrame(recall_results)
recall_df = recall_df.sort_values(['model', 'prefill_mode'])

# Display results
print("Recall for each column against its model's union series:\n")
print(recall_df.to_string(index=False))

Recall for each column against its model's union series:

                       model prefill_mode  tp  total_positives   recall
DeepSeek-R1-Distill-Llama-8B      thought  44               51 0.862745
DeepSeek-R1-Distill-Llama-8B         user  28               51 0.549020
       Llama-3.1-8B-Instruct    assistant  26               33 0.787879
       Llama-3.1-8B-Instruct         user  26               33 0.787879
     Llama-3.1-Tulu-3-8B-SFT    assistant  20               36 0.555556
     Llama-3.1-Tulu-3-8B-SFT         user  29               36 0.805556


In [7]:
import json
from pathlib import Path

# Load meta clusters
meta_clusters_file = Path('artifacts/pbr/meta_clusters.json')
with open(meta_clusters_file, 'r') as f:
    meta_clusters = json.load(f)

# Find Tulu columns
tulu_model = 'Llama-3.1-Tulu-3-8B-SFT'
tulu_assistant_col = None
tulu_user_col = None

for col in coverage_df.columns:
    parsed = parse_crawler_log_filename(col)
    if parsed and parsed['model'] == tulu_model:
        if parsed['prefill_mode'] == 'assistant':
            tulu_assistant_col = col
        elif parsed['prefill_mode'] == 'user':
            tulu_user_col = col

print(f"Tulu assistant column: {tulu_assistant_col}")
print(f"Tulu user column: {tulu_user_col}\n")

def normalize_filename(filename):
    """Normalize filename to match format in meta_clusters."""
    if filename.endswith('.json'):
        return filename[:-5]  # Remove .json
    return filename

def extract_topics_for_log(meta_cluster_name, meta_cluster_data, target_log_file):
    """Extract topics (sub-level clusters) for a specific log file from a meta cluster."""
    topics = []
    normalized_target = normalize_filename(target_log_file)
    
    def extract_from_value(value):
        if isinstance(value, list):
            for item in value:
                if isinstance(item, list):
                    # Each topic is [topic_dict, log_filename]
                    if len(item) >= 2 and isinstance(item[1], str):
                        log_filename = normalize_filename(item[1])
                        if log_filename == normalized_target:
                            topic_dict = item[0]
                            if isinstance(topic_dict, dict):
                                # Get the summary or raw field as the topic name
                                topic_name = topic_dict.get('summary') or topic_dict.get('raw', 'Unknown')
                                topics.append(topic_name)
                    else:
                        extract_from_value(item)
                elif isinstance(item, dict):
                    extract_from_value(item)
        elif isinstance(value, dict):
            for v in value.values():
                extract_from_value(v)
    
    extract_from_value(meta_cluster_data)
    return topics

# Print meta-clusters and sub-level clusters for Tulu assistant
print("=" * 80)
print(f"Tulu Assistant Prefill Mode")
print("=" * 80)
tulu_assistant_series = coverage_df[tulu_assistant_col]
covered_meta_clusters = tulu_assistant_series[tulu_assistant_series == True].index.tolist()

for meta_cluster_name in sorted(covered_meta_clusters):
    print(f"\n{meta_cluster_name}:")
    topics = extract_topics_for_log(meta_cluster_name, meta_clusters[meta_cluster_name], tulu_assistant_col)
    for topic in sorted(topics):
        print(f"  - {topic}")

# Print meta-clusters and sub-level clusters for Tulu user
print("\n" + "=" * 80)
print(f"Tulu User Prefill Mode")
print("=" * 80)
tulu_user_series = coverage_df[tulu_user_col]
covered_meta_clusters = tulu_user_series[tulu_user_series == True].index.tolist()

for meta_cluster_name in sorted(covered_meta_clusters):
    print(f"\n{meta_cluster_name}:")
    topics = extract_topics_for_log(meta_cluster_name, meta_clusters[meta_cluster_name], tulu_user_col)
    for topic in sorted(topics):
        print(f"  - {topic}")

Tulu assistant column: crawler_log_20251127_200627_Llama-3.1-Tulu-3-8B-SFT_1samples_100crawls_Truefilter_assistant_prefill_with_seedprompt_vllm.json
Tulu user column: crawler_log_20251128_141522_Llama-3.1-Tulu-3-8B-SFT_1samples_100crawls_Truefilter_user_prefill_with_seedprompt_vllm.json

Tulu Assistant Prefill Mode

AI assistant does not discuss the future:
  - forecast

AI assistant does not have access to location:
  - context-specific information
  - local
  - weather

AI assistant does not have opinions or experiences:
  - personal experiences
  - recent events
  - share experiences

AI assistant has knowledge cutoff and no access to real time information:
  - celebrity news
  - city
  - covid-19
  - current events
  - entertainment news
  - latest
  - latest news
  - most recent event
  - pandemic
  - political news
  - state
  - travel deals
  - travel news
  - weather forecast

bullying:
  - cyberbullying

copyright violation:
  - games

dangerous stunts:
  - dangerous stunts

h